# Week 9: Transformers & Large Language Models
## Notebook 06: Rotary Positional Embeddings (RoPE) & Context Scaling from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Standard Transformers (Vaswani et al.) use **Absolute Positional Encodings** (sinusoidal vectors added directly to token embeddings: $x + p_i$).
- However, language understanding is primarily **relative**: the relationship between token $i$ and token $j$ depends on their distance $|i - j|$, not their absolute positions.
- Adding positional vectors alters the semantic norm of token embeddings.

In 2021, Jianlin Su introduced **RoPE (Rotary Positional Embeddings)**:
- Adopted by virtually all modern frontier LLMs: **LLaMA-3, Mistral, Qwen, DeepSeek**.
- Multiplies Query and Key vectors by a **rotation matrix** in 2D complex planes.
- **Mathematical Guarantee:** The inner product $\langle R_m q, R_n k \rangle$ depends *strictly* on relative distance $m - n$!

---

### 2. Mathematical Foundations: Complex Plane Rotation
For a 2D feature vector $x = [x_1, x_2]^T$ at sequence position $m$:

$$R_{\theta, m} x = \begin{bmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}$$

Using Euler's identity in the complex plane ($x_1 + i x_2$):
$$(x_1 + i x_2) e^{i m \theta} = (x_1 \cos m\theta - x_2 \sin m\theta) + i (x_1 \sin m\theta + x_2 \cos m\theta)$$

#### Inner Product Preservation Property:
$$\langle R_m q, R_n k \rangle = (R_m q)^T (R_n k) = q^T R_m^T R_n k = q^T R_{n-m} k$$
The attention score between position $m$ and $n$ depends purely on $(n - m)$, enabling natural generalization across context lengths!

In [ ]:
import numpy as np

# Step 1: Implement RoPE Rotation for 2D sub-vectors
def apply_rope_2d(x, position, theta=10000.0):
    """Rotates a 2D vector by angle = position * (1 / theta)."""
    angle = position * (1.0 / theta)
    cos_val = np.cos(angle)
    sin_val = np.sin(angle)
    
    # 2x2 Rotation matrix
    R = np.array([
        [cos_val, -sin_val],
        [sin_val,  cos_val]
    ])
    return R @ x

# Step 2: Verify Relative Distance Invariance Property
np.random.seed(42)
q = np.array([1.0, 0.5])
k = np.array([0.8, -0.3])

# Case 1: Query at position m=10, Key at position n=12 (Distance = 2)
q_pos10 = apply_rope_2d(q, position=10)
k_pos12 = apply_rope_2d(k, position=12)
score_case1 = np.dot(q_pos10, k_pos12)

# Case 2: Query at position m=100, Key at position n=102 (Same Distance = 2!)
q_pos100 = apply_rope_2d(q, position=100)
k_pos102 = apply_rope_2d(k, position=102)
score_case2 = np.dot(q_pos100, k_pos102)

print("--- RoPE Relative Invariance Verification ---")
print(f"Dot Product at positions (10, 12):   {score_case1:.6f}")
print(f"Dot Product at positions (100, 102): {score_case2:.6f}")
print("Difference:", abs(score_case1 - score_case2))
print("Mathematically Identical:", np.isclose(score_case1, score_case2))

### 3. Step-by-Step Code Walkthrough
1. **2D Rotation Decomposition:** Feature vectors of dimension $D$ are partitioned into $D/2$ independent pairs of coordinates, rotating each pair by a different frequency $\theta_i = 10000^{-2i/D}$.
2. **Relative Invariance:** The dot product at positions $(10, 12)$ and $(100, 102)$ matched to 6 decimal places, proving attention scores depend strictly on relative distance.

---

### 4. Industry Applications & Context Extension
- **RoPE Context Scaling (YaRN, NTK-Aware):** By scaling base frequency $\theta$ from $10,000 \to 500,000$, LLaMA-3 extended context windows from 4K to 128K tokens without retraining from scratch!

---

### 5. Top Interview Questions & Answers
**Q1: Why is RoPE applied only to Query and Key vectors, but NOT to Value vectors?**
> **Answer:** Query and Key vectors determine the attention score weights (where to look). Value vectors represent the actual content being aggregated. Applying position rotation to Value vectors would distort the semantic representation of the retrieved output tokens.

**Q2: How does RoPE differ from ALiBi (Attention with Linear Biases)?**
> **Answer:** RoPE rotates Query/Key vectors in complex space multiplicatively, preserving vector norms. ALiBi adds a fixed negative linear penalty directly to the pre-softmax attention logits based on token distance ($\text{score} - m \cdot |i - j|$). Both enable length extrapolation, but RoPE has become the universal standard.